# M6P1 - Pipelines and Scripting for RNA-Seq

In this notebook, we will review Bash scripting and talk about how to apply it to our RNA-Seq pipelines, as well as briefly cover other workflows used in RNA-Seq processing.

**Objectives:**
- Review Bash scripting
- Practice some example scripts and job submission on the HPC cluster
- Discuss Nextflow, nf-core, and Galaxy for RNA-Seq processing

```{note}
**Runs on an HPC.** This notebook was written for the Hellbender HPC cluster (University of Missouri) and uses SLURM and/or large sequencing files. It is shown here for reading; saved outputs were removed for privacy. Set `REPO_DIR`, `DATA_DIR` and `WORK_DIR` (see the repository README) and download inputs with `data/fetch_data.sh` before running it on your own cluster.
```

## Bash review
So far, we have covered how to use a terminal and had some practice using the command-line; however, a majority of the scripting in this course has used `R`. It is important that we learn how to write Bash scripts to use in our RNA-Seq, or other NGS pipelines so that we can quickly parse, retrieve, modify, and utilize tools on our files. Generally, we will use bash scripts for our workflows, rather than typing in each command one by one. When we use `bash` within the HPC cluster, the following techniques are important: 
- **Bash scripts** for workflow automation  
- **Job schedulers** (Slurm, PBS, SGE) to manage compute jobs  
- **Modules** or **conda environments** for tool management  
- **Batch jobs** or **job arrays** to run multiple samples

### Let's recall some basics of `bash`: 

1. We use variables to store information within the shell. When we run scripts, we should store file names, locations, and parameters as such so that our scripts can be deployed and changed easily across different paths and environments. Below is an example of storing directories and parameters as a variable:

```bash
RAW_DIR="data/raw"
GENOME_DIR="references/genome"
THREADS=8
```
2. We can substitute our variables within our commands:
```bash
DATE=$(date +%Y-%m-%d)
echo "Today is $DATE, and the genome file is located in $GENOME_DIR"
```

3. `if` statements can be used for conditional options. This allows us to take certain actions within our code if a parameter is met. we use `if`, `elif` (else if), and `else` for these conditions. When done, we use finish, or `fi`. :
```bash
if [[ -f "$RAW_DIR/sample1.fastq.gz" ]]; then
    echo "File exists."
else
    echo "File missing."
fi
```
4. For loops can be used to run segments of code repeatedly until a condition is met. An example is running a command for each of the files within a directory:
```bash
for fq in data/raw/*.fastq.gz; do
    echo "Processing $fq"
done
```
5. Arrays allow us to handle multiple conditions, samples, or files within a set of commands (within nested `if` statements, in `while` loops, or in `for` loops). We can set these arrays manually:
```bash
SAMPLES=("sample1" "sample2" "sample3")
for s in "${SAMPLES[@]}"; do
    echo "Running $s"
done
```
Or we can detect these arrays based on a set of file names: 
```bash
SAMPLES=($(ls data/raw/*_R1.fastq.gz))
printf "%s\n" "${SAMPLES[@]}"
```
6. We can read from files, either CSV, TSV, or Excel, so that we can process data in a fast and efficient way:
```bash
CSV="samples.csv"

while IFS=',' read -r SAMPLE SRR GROUP; do
    echo "$SAMPLE uses SRR: $SRR"
done < <(tail -n +2 "$CSV")    # Skip header
```
7. Text parsing tools, such as `awk` and `grep`, can be used to search files quickly and parse their information. This can be piped to arrays for sample names, metadata sheets, or even to find links to retrieve files:
```bash
#basic search with grep
grep "AT1G" genes.gtf

#case insensitive search
grep -i "stress" annotation.txt

#using regex
grep -E "gene|exon" genes.gtf

#piping a command from grep into sam tools
samtools flagstat sample.bam | grep "mapped"
```

8. Commands can be added together, or piped using the `|` operator. A pipe sends the output of one command directly into the input of another command, without creating intermediate files.

```bash
# command1 | command2 | command3

#Count lines containing a word
grep "AT1G" genes.gtf | wc -l
#sort and keep unique items
cut -f 1 genes.gtf | sort | uniq
#Extract mapped reads from a BAM file
samtools view -h sample.bam | grep -v "unmapped" | samtools view -bS - > mapped.bam
```
9. Logging and debugging in bash is also easy, we can use commands such as `set` to control our logs and outputs in the case of an error within our bash script.
```bash
#print each command
set -x

#Stop on error and treat unset variables as errors
set -euo pipefail
```
10. A bash script is a file saved in `.sh` format that starts with the bash shebang (`#!bin/bash`). This tells the computer to use the bash interpreter to run the script:
```bash
#!bin/bash

module load subread

GTF="references/genes.gtf"
BAM_DIR="results/star"
OUT="results/counts/gene_counts.txt"

featureCounts \
    -T 8 \
    -a $GTF \
    -o $OUT \
    $BAM_DIR/*.bam
```

### Job Scheduling on the HPC cluster: 

When we use RSS OnDemand, we are running a job through a web interface. A job has 2 basic parts: the resources requested, and the script that it will run. This tells the HPC cluster how to allocate a set of computing resources and what to do with them. When we use RSS OnDemand, we ask for a set number of resources and tell the cluster to run a session, such as Jupyter Notebook. The same thing happens when we use the SLURM job scheduler, but we control each aspect of this from our terminal.

1. First, we connect to the login node for the Hellbender cluster via SSH within our local device's terminal
2. Then, we submit the bash script, which contains the needed resources and script we want to run.
3. This job runs according to the scheduler, and the outputs are saved where the script instructed.

There are two types of jobs within the HPC scheduler: 

Interactive: 
- You request resources and get a terminal or session immediately.
- Useful for testing, debugging, or running GUI-based tools (like Jupyter Notebook or RStudio).
- You can interact with the session in real time, run commands manually, and see outputs instantly.

Example command:
`srun --ntasks=1 --cpus-per-task=4 --mem=16G --time=01:00:00 --pty bash`

Scripted: 
- You submit a pre-written script to the scheduler, and it runs without direct interaction.
- Ideal for long-running analyses, pipelines, or processing multiple samples.
- Outputs are saved to files (stdout/stderr) as specified in the script.
- You don’t stay logged in while the job runs. 

Example: 
`sbatch my_rnaseq_pipeline.slurm`

To write a bash script for slurm: 

- Start with the shebang: `#!bin/bash`
- Specify the SLURM directives. SLURM directives start with #SBATCH and tell the scheduler what resources to allocate:
    ```bash
    #SBATCH --job-name=myjob        # Name of the job
    #SBATCH --ntasks=1              # Number of tasks
    #SBATCH --cpus-per-task=4       # Number of CPU cores per task
    #SBATCH --mem=16G               # Memory per node
    #SBATCH --time=02:00:00         # Max runtime (HH:MM:SS)
    #SBATCH --output=logs/%x_%j.out # Output file (%x = job name, %j = job ID)
    ```
- Load your modules or activate your environment:
    ```bash
    module load star
    # or
    conda activate rnaseq
    ```
- add the commands for the rest of your bash script:
    ```bash
    STAR --runThreadN 4 \
     --genomeDir references/star_index \
     --readFilesIn sample_R1.fastq.gz sample_R2.fastq.gz \
     --readFilesCommand zcat \
     --outSAMtype BAM SortedByCoordinate \
     --outFileNamePrefix results/sample_
     ```

Now we would put this all together: 

    ```bash
    #!bin/bash
    #SBATCH --job-name=myjob        # Name of the job
    #SBATCH --ntasks=1              # Number of tasks
    #SBATCH --cpus-per-task=4       # Number of CPU cores per task
    #SBATCH --mem=16G               # Memory per node
    #SBATCH --time=02:00:00         # Max runtime (HH:MM:SS)
    #SBATCH --output=logs/%x_%j.out # Output file (%x = job name, %j = job ID)
    
    module load star

    STAR --runThreadN 4 \
     --genomeDir references/star_index \
     --readFilesIn sample_R1.fastq.gz sample_R2.fastq.gz \
     --readFilesCommand zcat \
     --outSAMtype BAM SortedByCoordinate \
     --outFileNamePrefix results/sample_
    ```
Then, within the login node, we would navigate to the location of the script, and run it using `sbatch my_job.slurm`

You can read more about this on the [Slurm Documentation](https://slurm.schedmd.com/documentation.html) or the [Hellbender Documentation](https://docs.itrss.umsystem.edu/pub/hpc/hellbender)


### For a full tutorial on bash scripting you can use [this video](https://www.youtube.com/watch?v=tK9Oc6AEnR4)

## Practice writing bash scripts: 

This week, we will be running our own RNA-Seq analysis using STAR. You have done this a few times now, and it is time to apply all of the techniques you have learned so far on a new dataset [GSE72548 dataset](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE72548) for [Exercise M6E1](../exercise/M6E1_Full_RNA_Seq_Analysis.ipynb). Before we do this, we should practice writing scripts for things we have done before.

You will write two bash scripts that need to do the following: 

1. Write a bash script that will create a project folder called `m6_project`. This folder should be located within `Student_Folder/Module_6`.
- Follow the project file outline we have used in the past to create something like this:
```
    m6_project/
    ├── data
    │   ├── raw
    │   └── trimmed
    ├── references
    │   ├── annotation
    │   ├── genome
    │   ├── sra_info
    │   └── star_index
    ├── results
    │   ├── alignments
    │   ├── counts
    │   └── qc
    └── scripts
```
2. Retrieve the paired-end `fastq` files from the data set [GSE72548 dataset](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE72548) using the `fasterq` tool.
- Make sure to follow all the steps from M5L2. We need to gather the run info with `esearch`, parse the sample names with `awk` and `mapfile`, get their SRA links, and then create a CSV file wth the SRR links for each sample. After this, download them with `fasterq`.

**Both of these bash scripts should be saved within your `Module_6` folder.** You can run these scripts either using the SLURM scheduler using an SSH connection to the HPC cluster, or you can run them in the terminal within either Jupyter Notebook or the Hellbender Desktop. Make sure that the scripts work properly, and you can retrieve all the samples from the dataset as shown below: 


## Pipeline Managers for RNA-Seq and NGS pipelines

We have manually performed RNA-Seq several times now, and we will be working on creating our own pipeline for use on the Hellbender cluster. It is important to understand each step of the process in analyzing our data, so it is best to start from a manual run to learn the parameters and nuances of processing our data. That said, there are alternative options that reduce the workload, and we will discuss them more during the week of the final project. For now, here are some descriptions of the pipeline tools available for use in processing NGS data. 

We know processing RNA-Seq data involves multiple steps:

1. Quality control (FastQC, MultiQC)  
2. Adapter trimming (Trimmomatic, fastp)  
3. Alignment (STAR, HISAT2)  
4. Quantification (featureCounts, Salmon)  
5. Differential expression (DESeq2, edgeR)  

Because we have been managing these steps manually within Jupyter, it is clear that this can be error-prone and hard to reproduce. Workflow management systems automate, track, and scale analyses.

### **Nextflow** is a workflow management tool that allows:

- Portable pipelines across HPC, cloud, or local machines  
- Parallel execution of tasks with resource management  
- Reproducibility through Docker, Singularity, or Conda  

If desired, one could build their own pipeline for RNA-Seq using Nextflow, but many programs exist to do this in a more straightforward approach.

### [Nextflow documentation](https://www.nextflow.io/)

### **nf-core** is a community-built set of best-practice pipelines made to work within Nextflow.
- nf-core has ready-to-use RNA-Seq pipelines with a standardized input/output structure. These pipelines are built and updated constantly so that they are reliable and use the latest versions of tools.
- Heavy documentation within the program. There are lots of options for each pipeline, but it allows for flexible and reproducible analysis within the software
- Runs on top of Nextflow, so it can be used within containers.

![nf-core-rnaseq_metro_map_grey.png](images/M6P1_Pipelines_and_Scripting_for_RNA_Seq_dd0a14bf.png)

*Image: nf-core/rnaseq metro map, nf-core community, MIT License.*

### [nf-core documentation](https://nf-co.re/)
### [nf-core RNA-Seq pipeline](https://nf-co.re/rnaseq/3.22.0/)

### Galaxy
Galaxy is a web-based platform for bioinformatics workflow execution. It requires no programming, and if your organism of interest is supported offers a user-friendly way to process RNA-Seq data. 

### [Galaxy Project](https://galaxyproject.org/)

